# Supabase Database

Esta documentação detalha a arquitetura e o funcionamento do motor de persistência de dados criado para interagir com o **Supabase**, utilizando uma abordagem híbrida entre a API REST (via cliente oficial) e o acesso direto ao PostgreSQL (via `psycopg2`).

---

## Visão Geral

O código implementa uma camada de abstração de banco de dados dividida em duas responsabilidades principais: **Gerenciamento de Conexão** e **Motor de Consultas (Query Engine)**.

Diferente de abordagens que dependem apenas de bibliotecas de alto nível, este projeto foca na performance e no controle transacional robusto, permitindo execuções SQL puras, manipulação de JSONB e operações em lote (bulk operations) com segurança.

---

## Fluxo de Execução

O fluxo de uma operação comum segue estes passos:

1. **Inicialização:** O `SupabaseConnection` carrega as variáveis de ambiente e valida as credenciais.
2. **Lazy Loading:** A conexão com o banco PostgreSQL só é aberta no momento em que a primeira query é solicitada.
3. **Injeção de Dependência:** O `SupabaseQueryEngineer` recebe o método de conexão.
4. **Execução & Contexto:** O motor de consulta abre um cursor, executa o SQL e, em caso de sucesso, realiza o `commit`.
5. **Tratamento de Erro:** Se algo falhar, um `rollback` automático é acionado para manter a integridade dos dados.

---

## Tabela de Métodos

| **Classe** | **Método** | **Descrição Breve** |
| --- | --- | --- |
| **SupabaseConnection** | `get_connection()` | Retorna uma conexão ativa, reconectando se necessário. |
| **SupabaseConnection** | `close()` | Encerra a conexão física com o banco de dados. |
| **SupabaseQueryEngineer** | `select()` | Executa consultas de leitura e retorna uma lista de dicionários. |
| **SupabaseQueryEngineer** | `execute()` | Executa comandos de escrita (Insert, Update, Delete). |
| **SupabaseQueryEngineer** | `execute_many()` | Executa o mesmo comando SQL para uma lista de múltiplos parâmetros. |

---

## Arquitetura e Insights

- **Resiliência (Auto-reconexão):** O método `get_connection` realiza um "ping" (`SELECT 1`) no banco. Isso evita erros comuns de "broken pipe" em ambientes serverless ou quando a conexão fica ociosa por muito tempo.
- **Resultados Tipados:** O uso do `RealDictCursor` transforma as linhas do banco em dicionários Python nativos, facilitando o uso em APIs e manipulação de dados.
- **Segurança (SQL Injection):** O sistema utiliza parametrização (`%s`), delegando ao driver a limpeza dos dados, o que previne ataques de injeção de SQL.

---

## Documentação Detalhada das Classes

## Classe SupabaseConnection

**Descrição**

Gerencia o ciclo de vida da conexão com o ecossistema Supabase. Ela valida se o ambiente está configurado corretamente e fornece uma interface para obter conexões PostgreSQL estáveis através do protocolo TCP.

**Argumentos**

- Não possui argumentos de inicialização (lê diretamente de `.env`).

## Metodos

**1. get_connection**

- **Descrição:** Fornece uma instância de conexão `psycopg2`. Implementa o padrão Lazy Initialization.
- **Argumentos:** Nenhum.
- **Retornos:** `psycopg2.extensions.connection`
- **Raises:** `Exception` se as credenciais estiverem incorretas ou o banco estiver inacessível.
- **Exemplos:** `conn = connection.get_connection()`

---

## Classe SupabaseQueryEngineer

**Descrição**

O "cérebro" das operações de banco de dados. Esta classe isola a complexidade do SQL, gerenciando cursores e transações (Commit/Rollback) de forma transparente para o desenvolvedor.

**Argumentos**

- `get_connection` (Callable): Uma função ou método que retorne uma conexão ativa.

## Metodos

**1. select**

- **Descrição:** Executa queries de busca de dados.
- **Argumentos:** * `query` (str): Comando SQL SELECT.
    - `params` (tuple, opcional): Valores para substituir os placeholders `%s`.
- **Retornos:** `List[Dict[str, Any]]` - Uma lista onde cada item é uma linha da tabela.
- **Raises:** `Exception` em caso de erro de sintaxe SQL ou falha de conexão.
- **Exemplos:** `engine.select("SELECT * FROM users WHERE name = %s", ("Enzo",))`

**2. execute**

- **Descrição:** Executa comandos que alteram o estado do banco de dados (DDL ou DML).
- **Argumentos:**
    - `query` (str): Comando SQL (INSERT, UPDATE, DELETE, CREATE, etc).
    - `params` (tuple, opcional): Parâmetros da query.
- **Retornos:** `None`
- **Raises:** `Exception` com rollback automático em caso de falha.
- **Exemplos:** `engine.execute("UPDATE users SET name = %s", ("Novo Nome",))`

**3. execute_many**

- **Descrição:** Otimiza a inserção ou atualização de grandes volumes de dados.
- **Argumentos:**
    - `query` (str): Comando SQL parametrizado.
    - `params_list` (List[tuple]): Lista contendo as tuplas de parâmetros para cada repetição.
- **Retornos:** `None`
- **Raises:** `Exception` se qualquer uma das operações falhar (anula o lote inteiro).
- **Exemplos:** `engine.execute_many("INSERT INTO logs (msg) VALUES (%s)", [("Erro 1",), ("Erro 2",)])`
